1.LOAD DATASET

In [4]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

PLATFORM = "facebook"
SEEDS = [42, 123, 2024, 7, 99]
DATA_DIR = "../data_preprocess/processed_data/ml_data/"

results = []

for seed in SEEDS:
    train_df = pd.read_csv(f"{DATA_DIR}{PLATFORM}_train_seed{seed}.csv")
    test_df  = pd.read_csv(f"{DATA_DIR}{PLATFORM}_test_seed{seed}.csv")

    neg, pos = train_df["popularity"].value_counts()[0], train_df["popularity"].value_counts()[1]
    ratio = neg / pos

    X_train = train_df.drop(columns=["post_id", "user_id", "popularity"])
    y_train = train_df["popularity"]
    X_test  = test_df.drop(columns=["post_id", "user_id", "popularity"])
    y_test  = test_df["popularity"]

    model = XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=ratio, n_estimators=500, learning_rate=0.02,
        max_depth=6, min_child_weight=1, gamma=0.1,
        subsample=0.8, colsample_bytree=0.5,
        random_state=seed, n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "seed": seed,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    })
    print(f"[seed {seed}] done | F1={results[-1]['f1']:.4f} | ROC-AUC={results[-1]['roc_auc']:.4f}")

results_df = pd.DataFrame(results)
print("\n" + "="*60)
print(f"{PLATFORM.upper()} BASELINE — MEAN ± STD ACROSS {len(SEEDS)} SEEDS")
print("="*60)
print(results_df.set_index("seed"))
print("\nMean ± Std:")
summary = results_df.drop(columns="seed").agg(["mean", "std"])
print(summary)

[seed 42] done | F1=0.5919 | ROC-AUC=0.8573
[seed 123] done | F1=0.6232 | ROC-AUC=0.8761
[seed 2024] done | F1=0.6011 | ROC-AUC=0.8681
[seed 7] done | F1=0.6089 | ROC-AUC=0.8654
[seed 99] done | F1=0.5882 | ROC-AUC=0.8463

FACEBOOK BASELINE — MEAN ± STD ACROSS 5 SEEDS
      accuracy  precision    recall        f1   roc_auc
seed                                                   
42    0.799469   0.500000  0.725166  0.591892  0.857294
123   0.812085   0.521158  0.774834  0.623169  0.876064
2024  0.800797   0.502222  0.748344  0.601064  0.868108
7     0.808101   0.514874  0.745033  0.608931  0.865435
99    0.795485   0.493274  0.728477  0.588235  0.846324

Mean ± Std:
      accuracy  precision    recall        f1   roc_auc
mean  0.803187   0.506306  0.744371  0.602658  0.862645
std   0.006749   0.011405  0.019785  0.014022  0.011322


In [3]:
results_df.insert(0, "model", "baseline_metadata")
results_df.insert(0, "platform", PLATFORM)

import os
os.makedirs("../results", exist_ok=True)
results_df.to_csv(f"../results/{PLATFORM}_baseline_metadata.csv", index=False)
print(f"Saved: ../results/{PLATFORM}_baseline_metadata.csv")

Saved: ../results/facebook_baseline_metadata.csv
